# 11_embedded: interactive lesson

Read README.md first. Predict each outcome before executing. Code below is mirrored from the scripts; edit a copy to experiment. No model downloads or containers are started.

In [ ]:
import os
import sys

from common.settings import ROOT

os.chdir(ROOT)
sys.argv = ["notebook"]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["ANONYMIZED_TELEMETRY"] = "False"

## 01_from_scratch

Source: `modules/11_embedded/examples/01_from_scratch.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
"""Model logical backup as a reproducible catalog export.

Run: uv run python modules/11_embedded/examples/01_from_scratch.py
Concepts: Model logical backup as a reproducible catalog export.
Expected: original ranked IDs after restore.
CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
"""

import tempfile
from pathlib import Path

from common.data import records_from_products
from common.stores.types import CollectionSpec
from common.teaching import catalog
from tinyvec import TinyVec


def main() -> None:
    # KEY IDEA: Model logical backup as a reproducible catalog export.
    products, matrix, q, meta = catalog(80)
    store = TinyVec()
    store.create_collection(CollectionSpec("tiny", matrix.shape[1], "cosine", meta["space"]))
    store.upsert(records_from_products(products, matrix))
    with tempfile.TemporaryDirectory() as tmp:
        store.save(Path(tmp))
        before = store.search(q, 3)
        store.delete([before[0].id])
        restored = TinyVec.load(Path(tmp)).search(q, 3)
        assert [h.id for h in before] == [h.id for h in restored]
        print("restored IDs", [h.id for h in restored])


if __name__ == "__main__":
    main()

## 02_production

Source: `modules/11_embedded/examples/02_production.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Update a Chroma listing, reopen the local file and query its metadata.

    Run: uv run --extra embedded python modules/11_embedded/examples/02_production.py
    Concepts: Update a Chroma listing, reopen the local file and query its metadata.
    Expected: Chroma reopened IDs and updated region.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
    """

    # course-extra: embedded
    import uuid

    import chromadb

    from common.settings import cache_dir
    from common.teaching import catalog

    def main() -> None:
        # KEY IDEA: Update a Chroma listing, reopen the local file and query its metadata.
        path = str(cache_dir() / "tier3/chroma")
        products, matrix, q, _ = catalog(100)
        name = "native_" + uuid.uuid4().hex
        with chromadb.PersistentClient(
            path=path, settings=chromadb.Settings(anonymized_telemetry=False)
        ) as db:
            table = db.create_collection(
                name, embedding_function=None, configuration={"hnsw": {"space": "cosine"}}
            )
            table.upsert(
                ids=products.id.tolist(),
                embeddings=matrix.tolist(),
                documents=products.title.tolist(),
                metadatas=[{"region": r} for r in products.region],
            )
            key = products.id.iloc[0]
            table.update(ids=[key], metadatas=[{"region": "special"}])
        with chromadb.PersistentClient(
            path=path, settings=chromadb.Settings(anonymized_telemetry=False)
        ) as db:
            table = db.get_collection(name, embedding_function=None)
            try:
                hit = table.query(
                    query_embeddings=[q.tolist()], where={"region": "special"}, n_results=1
                )
                print(hit["ids"], table.get(ids=[key])["documents"])
                assert hit["ids"] == [[key]]
                table.delete(ids=[key])
                assert table.count() == 99
            finally:
                db.delete_collection(name)

    if __name__ == "__main__":
        main()

## 03_measure

Source: `modules/11_embedded/examples/03_measure.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Measure a controlled database comparison on the same catalog.

    Run: uv run --extra embedded python modules/11_embedded/examples/03_measure.py
    Concepts: Measure a controlled database comparison on the same catalog.
    Expected: reproducible local benchmark.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
    """

    # course-extra: embedded
    from common.tier3 import measure

    def main() -> None:
        # KEY IDEA: Measure a controlled database comparison on the same catalog.
        for backend in ("chroma", "lancedb"):
            run = measure(backend)
            print(backend, run["summary"]["p95_ms"])

    if __name__ == "__main__":
        main()

## 04_lancedb

Source: `modules/11_embedded/examples/04_lancedb.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Compare Lance table versions, native hybrid ranking and vector indexing.

    Run: uv run --extra embedded python modules/11_embedded/examples/04_lancedb.py
    Concepts: Compare Lance table versions, native hybrid ranking and vector indexing.
    Expected: restored version, IVF-PQ and northern hybrid IDs.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
    """

    # course-extra: embedded
    import tempfile

    import lancedb
    import pyarrow as pa
    from lancedb.index import FTS, IvfPq

    from common.teaching import catalog

    def main() -> None:
        # KEY IDEA: Compare Lance table versions, native hybrid ranking and vector indexing.
        products, matrix, q, _ = catalog(640)
        schema = pa.schema(
            [
                ("id", pa.string()),
                ("text", pa.string()),
                ("region", pa.string()),
                ("v", pa.list_(pa.float32(), matrix.shape[1])),
            ]
        )
        rows = [
            {"id": p.id, "text": p.title, "region": p.region, "v": v.tolist()}
            for p, v in zip(products.itertuples(), matrix, strict=True)
        ]
        with tempfile.TemporaryDirectory() as tmp:
            with lancedb.connect(tmp) as db:
                table = db.create_table("products", data=pa.Table.from_pylist(rows, schema=schema))
                version = table.version
                table.delete("region = 'north'")
                smaller = table.count_rows()
                table.checkout(version)
                assert smaller < table.count_rows() == 640
                table.restore()
                table.create_index("text", config=FTS())
                table.create_index(
                    "v",
                    config=IvfPq(
                        distance_type="cosine", num_partitions=4, num_sub_vectors=8, num_bits=4
                    ),
                )
                hits = (
                    table.search(query_type="hybrid", vector_column_name="v")
                    .vector(q)
                    .text("mug warmer")
                    .where("region = 'north'", prefilter=True)
                    .limit(5)
                    .to_list()
                )
                assert hits and all(h["region"] == "north" for h in hits)
                print("versions", table.list_versions(), "hybrid hits", [h["id"] for h in hits])
                del table

    if __name__ == "__main__":
        main()

## 05_sqlite_vec

Source: `modules/11_embedded/examples/05_sqlite_vec.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Join native SQLite vector scores to catalog metadata.

    Run: uv run --extra embedded python modules/11_embedded/examples/05_sqlite_vec.py
    Concepts: Join native SQLite vector scores to catalog metadata.
    Expected: five SQLite distances.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
    """

    # course-extra: embedded
    import sqlite3

    import sqlite_vec

    from common.teaching import catalog

    def main() -> None:
        # KEY IDEA: Join native SQLite vector scores to catalog metadata.
        rows, matrix, q, _ = catalog(80)
        with sqlite3.connect(":memory:") as db:
            db.enable_load_extension(True)
            sqlite_vec.load(db)
            db.enable_load_extension(False)
            db.execute("CREATE TABLE products(id TEXT PRIMARY KEY, region TEXT, v BLOB)")
            db.executemany(
                "INSERT INTO products VALUES (?,?,?)",
                [
                    (p.id, p.region, v.tobytes())
                    for p, v in zip(rows.itertuples(), matrix, strict=True)
                ],
            )
            hits = db.execute(
                "SELECT id,vec_distance_cosine(v,?) FROM products WHERE region=? ORDER BY 2,id LIMIT 5",
                (q.tobytes(), "north"),
            ).fetchall()
            print(hits)
            assert hits and all(rows.set_index("id").loc[k, "region"] == "north" for k, _ in hits)

    if __name__ == "__main__":
        main()

## 06_duckdb

Source: `modules/11_embedded/examples/06_duckdb.py`. What do you expect? Which assertion could catch a wrong implementation?

In [ ]:
import importlib.util

if importlib.util.find_spec("chromadb") is None:
    print("SKIP native comparison: --extra embedded")
else:
    """Compare exact array SQL with an explicitly prepared VSS index.

    Run: uv run --extra embedded python modules/11_embedded/examples/06_duckdb.py
    Concepts: Compare exact array SQL with an explicitly prepared VSS index.
    Expected: exact SQL and optional HNSW scan.
    CPU: about 2–90 seconds, depending on the profile and index. Models: included vectors; optional local caption model only for the generation example. Marlowe local inputs; no database server required.
    """

    # course-extra: embedded
    import duckdb

    from common.settings import cache_dir
    from common.teaching import catalog

    def main() -> None:
        # KEY IDEA: Compare exact array SQL with an explicitly prepared VSS index.
        products, matrix, q, _ = catalog(160)
        ext = cache_dir() / "tier3/duckdb-extensions"
        ext.mkdir(parents=True, exist_ok=True)
        with duckdb.connect(
            config={
                "extension_directory": str(ext),
                "autoload_known_extensions": False,
                "autoinstall_known_extensions": False,
            }
        ) as db:
            db.execute("CREATE TABLE p(id VARCHAR,v FLOAT[384])")
            db.executemany(
                "INSERT INTO p VALUES (?,?)", list(zip(products.id, matrix.tolist(), strict=True))
            )
            sql = "SELECT id FROM p ORDER BY array_cosine_distance(v,?::FLOAT[384]) LIMIT 5"
            exact = db.execute(sql, [q.tolist()]).fetchall()
            print("SQL exact", exact)
            if db.execute(
                "SELECT installed FROM duckdb_extensions() WHERE extension_name='vss'"
            ).fetchone()[0]:
                db.execute("LOAD vss")
                db.execute("CREATE INDEX vectors ON p USING HNSW(v) WITH (metric='cosine')")
                print(
                    db.execute("EXPLAIN " + sql, [q.tolist()]).fetchone()[1],
                    db.execute(sql, [q.tolist()]).fetchall(),
                )
            else:
                print(
                    "VSS optional: run scripts/prepare_duckdb_vss.py to install its signed extension"
                )
            assert len(exact) == 5

    if __name__ == "__main__":
        main()

## Change one thing

Explain the result before and after changing the input. The code below is deliberately small enough to inspect.

In [ ]:
from common.teaching import catalog

rows, _, _, _ = catalog(100)
print(rows.groupby("region").price.median())
# Change the group or price threshold, then repeat the native comparison for 11_embedded.

## Finish

Complete Practice, Apply, Stretch, and the six-question quiz in README.md. No notebook output is committed. Preview `scripts/cleanup.py` before deleting generated course files.